#### Importing Libraries

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split

#### Loading the Dataset

In [2]:
data_path = Path("../data/raw/20000-Utterances-Training-dataset-for-chatbots-virtual-assistant-Bitext-sample.csv")

df = pd.read_csv(data_path)

print(f"Dataset Shape: {df.shape}")
display(df.head())

Dataset Shape: (21534, 4)


,flags,utterance,category,intent
0,BILC,"I don't have an online account, what do I have...",ACCOUNT,create_account
1,BILQZ,can you tell me if i can regisger two accounts...,ACCOUNT,create_account
2,BPLC,"I have no online account, open one, please",ACCOUNT,create_account
3,BIPLD,"could you ask an agent how to open an account,...",ACCOUNT,create_account
4,BLQC,"i want an online account, create one",ACCOUNT,create_account


#### Removing Duplicate Customer Utterances

In [4]:
initial_rows = len(df)

df = df.drop_duplicates(subset=["utterance"], keep="first").copy()
df.reset_index(drop=True)

removed_rows = initial_rows - len(df)

print(f"Original Rows: {initial_rows:,}")
print(f"Duplicate Rows Removed: {removed_rows:,}")
print(f"Remaining Rows: {len(df):,}")
print(f"Remaining Duplicate Utterances: {df.duplicated(subset=['utterance']).sum()}")

Original Rows: 21,513
Duplicate Rows Removed: 0
Remaining Rows: 21,513
Remaining Duplicate Utterances: 0


#### Observations
- **21 duplicate customer utterances** were removed, reducing the dataset from 21,534 to 21,513 records.
- **No duplicate utterances remain**, ensuring that identical customer queries cannot appear across training and testing datasets.
- **Decision:** The deduplicated dataset will be used for subsequent data validation, feature preparation, and stratified splitting.

#### Validating Missing Values and Target Labels

In [5]:
print("Missing Values:")
display(df[["utterance", "intent", "category"]].isnull().sum())

print(f"\nEmpty Utterances: {df['utterance'].str.strip().eq('').sum()}")
print(f"Unique Intents: {df['intent'].nunique()}")
print(f"Unique Categories: {df['category'].nunique()}")

intent_category_check = df.groupby("intent")["category"].nunique()

print(f"Intents Assigned to Multiple Categories: {(intent_category_check > 1).sum()}")

Missing Values:


utterance    0
intent       0
category     0
dtype: int64


Empty Utterances: 0
Unique Intents: 27
Unique Categories: 11
Intents Assigned to Multiple Categories: 0


#### Observations
- **No empty customer utterances** were detected in the cleaned dataset.
- **All 27 intents and 11 categories are retained**, with each intent mapped consistently to a single category.
- **Decision:** The dataset is suitable for feature and target selection, subject to confirming zero missing values.

#### Selecting Input Features and Target Variable

In [6]:
X = df["utterance"].copy()
y = df["intent"].copy()

print(f"Input Feature: {X.name}")
print(f"Target Variable: {y.name}")
print(f"Total Samples: {len(X):,}")
print(f"Number of Target Classes: {y.nunique()}")

display(y.value_counts().rename("Sample Count").to_frame())

Input Feature: utterance
Target Variable: intent
Total Samples: 21,513
Number of Target Classes: 27


,Sample Count
intent,
payment_issue,4365
create_account,2122
contact_customer_service,2055
get_invoice,1421
track_order,1219
get_refund,1150
contact_human_agent,1026
check_invoices,1013
recover_password,986


#### Observations
- **`utterance` is selected as the input feature**, while `intent` is selected as the target variable for multiclass classification.
- **All 27 intent classes are retained**, with `payment_issue` (4,365) being the largest class and `cancel_order` (36) the smallest.
- **Decision:** A stratified train-test split will be applied to preserve class proportions, while `category` and `flags` are excluded from predictive features.

#### Creating Stratified Training and Testing Datasets

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f"Training Samples: {len(X_train):,}")
print(f"Testing Samples: {len(X_test):,}")
print(f"Training Percentage: {len(X_train) / len(X) * 100:.2f}%")
print(f"Testing Percentage: {len(X_test) / len(X) * 100:.2f}%")

print(f"\nTraining Classes: {y_train.nunique()}")
print(f"Testing Classes: {y_test.nunique()}")

print(f"\nSmallest Training Class: {y_train.value_counts().min()}")
print(f"Smallest Testing Class: {y_test.value_counts().min()}")

Training Samples: 17,210
Testing Samples: 4,303
Training Percentage: 80.00%
Testing Percentage: 20.00%

Training Classes: 27
Testing Classes: 27

Smallest Training Class: 29
Smallest Testing Class: 7


#### Observations
- **17,210 training samples (80%)** and **4,303 testing samples (20%)** were created using stratified sampling.
- **All 27 intent classes are represented in both datasets**, with the smallest class containing 29 training samples and 7 testing samples.
- **Decision:** The training set will be used for feature engineering and model development, while the test set will remain untouched until final evaluation.

#### Saving Cleaned and Stratified Datasets

In [8]:
processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

train_df = pd.DataFrame({
    "utterance": X_train,
    "intent": y_train
})

test_df = pd.DataFrame({
    "utterance": X_test,
    "intent": y_test
})

df.to_csv(processed_dir / "cleaned_customer_intents.csv", index=False)
train_df.to_csv(processed_dir / "train.csv", index=False)
test_df.to_csv(processed_dir / "test.csv", index=False)

print("Processed datasets saved successfully.\n")

for filename in [
    "cleaned_customer_intents.csv",
    "train.csv",
    "test.csv"
]:
    filepath = processed_dir / filename
    print(f"{filename}: {filepath.stat().st_size / 1024:.2f} KB")

print(f"\nTraining Dataset Shape: {train_df.shape}")
print(f"Testing Dataset Shape: {test_df.shape}")

Processed datasets saved successfully.

cleaned_customer_intents.csv: 1775.68 KB
train.csv: 1197.52 KB
test.csv: 298.84 KB

Training Dataset Shape: (17210, 2)
Testing Dataset Shape: (4303, 2)


#### Observations
- **The cleaned dataset and stratified train-test splits** were successfully saved in `data/processed/`.
- **The training dataset contains 17,210 records**, while the testing dataset contains 4,303 records, each with an utterance and its corresponding intent.
- **Decision:** Saved datasets will serve as reusable inputs for feature engineering, model development, and evaluation, ensuring consistency across notebooks.

#### Verifying Saved Dataset Integrity

In [9]:
saved_train = pd.read_csv(processed_dir / "train.csv")
saved_test = pd.read_csv(processed_dir / "test.csv")
saved_cleaned = pd.read_csv(processed_dir / "cleaned_customer_intents.csv")

print(f"Cleaned Dataset Shape: {saved_cleaned.shape}")
print(f"Training Dataset Shape: {saved_train.shape}")
print(f"Testing Dataset Shape: {saved_test.shape}")

print(f"\nMissing Training Values: {saved_train.isnull().sum().sum()}")
print(f"Missing Testing Values: {saved_test.isnull().sum().sum()}")

print(f"\nTraining Classes: {saved_train['intent'].nunique()}")
print(f"Testing Classes: {saved_test['intent'].nunique()}")

overlapping_utterances = set(saved_train["utterance"]) & set(saved_test["utterance"])
print(f"\nOverlapping Train-Test Utterances: {len(overlapping_utterances)}")

Cleaned Dataset Shape: (21513, 4)
Training Dataset Shape: (17210, 2)
Testing Dataset Shape: (4303, 2)

Missing Training Values: 0
Missing Testing Values: 0

Training Classes: 27
Testing Classes: 27

Overlapping Train-Test Utterances: 0


#### Observations
- **All saved datasets were reloaded successfully**, retaining 21,513 cleaned records, 17,210 training records, and 4,303 testing records.
- **No missing values or overlapping utterances** were detected between the saved training and testing datasets.
- **Key Finding:** All 27 intent classes are preserved across both splits, confirming that the saved datasets are ready for feature engineering and subsequent model development.